# Pseudobulk model building — flashier

**Environment:** `clamp-analyses`

Empirical Bayes matrix factorization (point-exponential × point-normal priors) on every pseudobulk dataset. Preprocesses raw counts from `bulk_expr.csv`. Reads `k.csv` from CLAMP output for rank. Outputs written to `output/01_model_building/05_pseudobulk/<dataset>/flashier/`.

## Libraries

In [1]:
library(data.table)
library(here)
library(CLAMP)
library(PCAtools)
library(rsvd)
library(flashier)
library(ebnm)

set.seed(123)


here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



Loading required package: ggplot2



Loading required package: ggrepel




Attaching package: ‘PCAtools’




The following objects are masked from ‘package:stats’:

    biplot, screeplot




Loading required package: ebnm



## Configuration

In [2]:
DATASET         = "PBMC_Perez2022"
BACKFIT_MAXITER = 10L
OUT_ROOT        = "output/01_model_building/05_pseudobulk"
DATA_DIR        = "data/pseudobulk"

## flashier helper

In [3]:
myflash <- function(data, ebnm_fn = ebnm_point_normal, var_type = 0L,
                    greedy_Kmax = 50L, backfit = FALSE,
                    backfit_maxiter = 20L, verbose = 0L) {
  fl <- flash_init(data, var_type = var_type)
  fl <- flash_greedy(fl, Kmax = greedy_Kmax, ebnm_fn = ebnm_fn, verbose = verbose)
  if (backfit) fl <- flash_backfit(fl, verbose = verbose, maxiter = backfit_maxiter)
  fl
}

## Build flashier model for each dataset

In [4]:
message("========== ", DATASET, " ==========")
out_dir <- file.path(here(), OUT_ROOT, DATASET)

# Load preprocessed data
norm_dt    <- fread(file.path(here(), OUT_ROOT, DATASET, "norm.csv"))
norm_genes <- norm_dt[[1]]
norm       <- as.matrix(norm_dt[, -1, with = FALSE])
storage.mode(norm) <- "numeric"
rownames(norm) <- norm_genes
samples <- colnames(norm)
cat(DATASET, "norm:", nrow(norm), "genes x", ncol(norm), "samples\n")

# Load k
k <- as.integer(read.csv(file.path(here(), OUT_ROOT, DATASET, "k.csv"))$k[1])
message("  k = ", k)

# flashier expects samples x genes
X <- t(norm)
stopifnot(k <= min(nrow(X), ncol(X)))

# Run flashier
message("  Running flashier (greedy_Kmax=", k, ", backfit_maxiter=", BACKFIT_MAXITER, ") ...")
fl <- myflash(
  data            = X,
  ebnm_fn         = c(ebnm_point_exponential, ebnm_point_normal),
  greedy_Kmax     = k,
  backfit         = TRUE,
  backfit_maxiter = BACKFIT_MAXITER,
  verbose         = 0L
)

# data = samples x genes -> L_pm = samples x k, F_pm = genes x k
B <- t(fl$L_pm)
colnames(B) <- samples
rownames(B) <- paste0("LV", seq_len(nrow(B)))

Z_mat <- fl$F_pm
rownames(Z_mat) <- norm_genes
colnames(Z_mat) <- paste0("LV", seq_len(ncol(Z_mat)))

message("  Factors retained: ", nrow(B))

model_dir <- file.path(out_dir, "flashier")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
write.csv(as.data.frame(B),     file.path(model_dir, "B.csv"))
write.csv(as.data.frame(Z_mat), file.path(model_dir, "Z.csv"))
saveRDS(fl, file.path(model_dir, "flashier_model.rds"))
message("  flashier saved -> ", model_dir)

========== Lung_Sikkema2023 ==========



Lung_Sikkema2023 norm: 17145 genes x 100 samples


  k = 18



  Running flashier (greedy_Kmax=18, backfit_maxiter=10) ...



Warning message in report.maxiter.reached(verbose.lvl):
“Maximum number of iterations reached.”


  Factors retained: 18



  flashier saved -> /home/msubirana/Documents/pivlab/clamp-analyses/output/01_model_building/05_pseudobulk/Lung_Sikkema2023/flashier

